In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score
from statsmodels.stats.contingency_tables import mcnemar

In [ ]:
BASE_DIR = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison"
)

CT_RESULTS_PATH = os.path.join(
    BASE_DIR,
    "CT",
    "Results",
    "CT_Matched_Holdout_vs_OOF_Results.csv"
)

CIFAR_RESULTS_PATH = os.path.join(
    BASE_DIR,
    "CIFAR10",
    "Results",
    "CIFAR10_Matched_Holdout_vs_OOF_Results.csv"
)

CT_PRED_DIR = os.path.join(
    BASE_DIR,
    "CT",
    "Test_Predictions"
)

CIFAR_PRED_DIR = os.path.join(
    BASE_DIR,
    "CIFAR10",
    "Test_Predictions"
)

FINAL_DIR = os.path.join(
    BASE_DIR,
    "Combined_Paper_Results"
)

os.makedirs(FINAL_DIR, exist_ok=True)

print("CT results:", CT_RESULTS_PATH)
print("CIFAR results:", CIFAR_RESULTS_PATH)
print("Final output:", FINAL_DIR)

In [ ]:
if not os.path.exists(CT_RESULTS_PATH):
    raise FileNotFoundError(
        f"CT results not found:\n{CT_RESULTS_PATH}"
    )

if not os.path.exists(CIFAR_RESULTS_PATH):
    raise FileNotFoundError(
        f"CIFAR-10 results not found:\n{CIFAR_RESULTS_PATH}"
    )

ct_results = pd.read_csv(CT_RESULTS_PATH)
cifar_results = pd.read_csv(CIFAR_RESULTS_PATH)

combined_results = pd.concat(
    [ct_results, cifar_results],
    ignore_index=True
)

combined_results.to_csv(
    os.path.join(
        FINAL_DIR,
        "Paper1_Combined_Matched_Results.csv"
    ),
    index=False
)

display(combined_results)

In [ ]:
accuracy_table = combined_results[
    [
        "Dataset",
        "Protocol",
        "Method",
        "Accuracy",
        "Macro_F1"
    ]
].copy()

accuracy_table["Accuracy_Percent"] = (
    accuracy_table["Accuracy"] * 100
)

accuracy_table["Macro_F1_Percent"] = (
    accuracy_table["Macro_F1"] * 100
)

accuracy_table = accuracy_table[
    [
        "Dataset",
        "Protocol",
        "Method",
        "Accuracy_Percent",
        "Macro_F1_Percent"
    ]
]

accuracy_table[
    "Accuracy_Percent"
] = accuracy_table[
    "Accuracy_Percent"
].round(2)

accuracy_table[
    "Macro_F1_Percent"
] = accuracy_table[
    "Macro_F1_Percent"
].round(2)

accuracy_table.to_csv(
    os.path.join(
        FINAL_DIR,
        "Paper1_Combined_Accuracy_Table.csv"
    ),
    index=False
)

display(accuracy_table)

In [ ]:
gain_rows = []

for dataset in combined_results["Dataset"].unique():

    dataset_df = combined_results[
        combined_results["Dataset"] == dataset
    ]

    for method in [
        "LogisticRegression",
        "LightGBM"
    ]:

        holdout_row = dataset_df[
            (
                dataset_df["Protocol"]
                == "Fixed hold-out"
            )
            &
            (
                dataset_df["Method"]
                == method
            )
        ]

        oof_row = dataset_df[
            (
                dataset_df["Protocol"]
                == "Five-fold OOF"
            )
            &
            (
                dataset_df["Method"]
                == method
            )
        ]

        if (
            len(holdout_row) == 1
            and len(oof_row) == 1
        ):
            holdout_accuracy = float(
                holdout_row["Accuracy"].iloc[0]
            )

            oof_accuracy = float(
                oof_row["Accuracy"].iloc[0]
            )

            gain_rows.append(
                {
                    "Dataset": dataset,
                    "Method": method,
                    "Holdout_Accuracy": holdout_accuracy,
                    "OOF_Accuracy": oof_accuracy,
                    "OOF_Gain_Percentage_Points": (
                        oof_accuracy
                        - holdout_accuracy
                    ) * 100
                }
            )

protocol_gains = pd.DataFrame(gain_rows)

protocol_gains[
    "Holdout_Accuracy"
] = (
    protocol_gains[
        "Holdout_Accuracy"
    ] * 100
).round(2)

protocol_gains[
    "OOF_Accuracy"
] = (
    protocol_gains[
        "OOF_Accuracy"
    ] * 100
).round(2)

protocol_gains[
    "OOF_Gain_Percentage_Points"
] = protocol_gains[
    "OOF_Gain_Percentage_Points"
].round(2)

protocol_gains.to_csv(
    os.path.join(
        FINAL_DIR,
        "Paper1_Protocol_Gains.csv"
    ),
    index=False
)

display(protocol_gains)

In [ ]:
plot_df = combined_results.copy()

plot_df["Label"] = (
    plot_df["Dataset"]
    + " | "
    + plot_df["Protocol"]
    + " | "
    + plot_df["Method"]
)

plot_df["Accuracy_Percent"] = (
    plot_df["Accuracy"] * 100
)

plt.figure(figsize=(13, 7))

bars = plt.bar(
    plot_df["Label"],
    plot_df["Accuracy_Percent"]
)

for bar, value in zip(
    bars,
    plot_df["Accuracy_Percent"]
):
    plt.text(
        bar.get_x()
        + bar.get_width() / 2,
        bar.get_height() + 0.08,
        f"{value:.2f}%",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

plt.ylabel("Accuracy (%)")
plt.xlabel("Dataset, protocol and method")
plt.title(
    "Matched hold-out versus five-fold OOF stacking"
)

plt.xticks(
    rotation=35,
    ha="right"
)

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.4
)

minimum_value = plot_df[
    "Accuracy_Percent"
].min()

maximum_value = plot_df[
    "Accuracy_Percent"
].max()

plt.ylim(
    minimum_value - 2,
    min(100, maximum_value + 1.5)
)

plt.tight_layout()

combined_chart_path = os.path.join(
    FINAL_DIR,
    "Paper1_Combined_Accuracy_Comparison.png"
)

plt.savefig(
    combined_chart_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved:", combined_chart_path)

In [ ]:
def load_prediction_file(
    prediction_dir,
    filename
):
    path = os.path.join(
        prediction_dir,
        filename
    )

    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Prediction file not found:\n{path}"
        )

    return np.load(path).reshape(-1)


def run_mcnemar_test(
    y_true,
    pred_a,
    pred_b,
    method_a,
    method_b,
    dataset
):
    correct_a = pred_a == y_true
    correct_b = pred_b == y_true

    # A correct, B wrong
    b = int(
        np.sum(
            correct_a
            & (~correct_b)
        )
    )

    # A wrong, B correct
    c = int(
        np.sum(
            (~correct_a)
            & correct_b
        )
    )

    table = [
        [
            int(
                np.sum(
                    correct_a & correct_b
                )
            ),
            b
        ],
        [
            c,
            int(
                np.sum(
                    (~correct_a)
                    & (~correct_b)
                )
            )
        ]
    ]

    exact_test = (
        b + c < 25
    )

    result = mcnemar(
        table,
        exact=exact_test,
        correction=not exact_test
    )

    return {
        "Dataset": dataset,
        "Method_A": method_a,
        "Method_B": method_b,
        "A_Correct_B_Wrong": b,
        "A_Wrong_B_Correct": c,
        "Statistic": float(result.statistic),
        "P_Value": float(result.pvalue),
        "Significant_0.05": (
            result.pvalue < 0.05
        )
    }

In [ ]:
ct_true = load_prediction_file(
    CT_PRED_DIR,
    "CT_Test_True_Labels.npy"
)

ct_predictions = {
    "Soft Voting": load_prediction_file(
        CT_PRED_DIR,
        "SoftVoting_Predicted_Labels.npy"
    ),
    "Hold-out LR": load_prediction_file(
        CT_PRED_DIR,
        "Holdout_LogisticRegression_Predicted_Labels.npy"
    ),
    "Hold-out LightGBM": load_prediction_file(
        CT_PRED_DIR,
        "Holdout_LightGBM_Predicted_Labels.npy"
    ),
    "OOF LR": load_prediction_file(
        CT_PRED_DIR,
        "OOF_LogisticRegression_Predicted_Labels.npy"
    ),
    "OOF LightGBM": load_prediction_file(
        CT_PRED_DIR,
        "OOF_LightGBM_Predicted_Labels.npy"
    )
}

ct_comparisons = [
    ("Soft Voting", "OOF LR"),
    ("Soft Voting", "OOF LightGBM"),
    ("Hold-out LR", "OOF LR"),
    (
        "Hold-out LightGBM",
        "OOF LightGBM"
    ),
    ("OOF LR", "OOF LightGBM")
]

ct_test_rows = []

for method_a, method_b in ct_comparisons:
    ct_test_rows.append(
        run_mcnemar_test(
            y_true=ct_true,
            pred_a=ct_predictions[method_a],
            pred_b=ct_predictions[method_b],
            method_a=method_a,
            method_b=method_b,
            dataset="CT"
        )
    )

ct_mcnemar_results = pd.DataFrame(
    ct_test_rows
)

display(ct_mcnemar_results)

In [ ]:
cifar_true = load_prediction_file(
    CIFAR_PRED_DIR,
    "CIFAR10_Test_True_Labels.npy"
)

cifar_predictions = {
    "Soft Voting": load_prediction_file(
        CIFAR_PRED_DIR,
        "SoftVoting_Predicted_Labels.npy"
    ),
    "Hold-out LR": load_prediction_file(
        CIFAR_PRED_DIR,
        "Holdout_LogisticRegression_Predicted_Labels.npy"
    ),
    "Hold-out LightGBM": load_prediction_file(
        CIFAR_PRED_DIR,
        "Holdout_LightGBM_Predicted_Labels.npy"
    ),
    "OOF LR": load_prediction_file(
        CIFAR_PRED_DIR,
        "OOF_LogisticRegression_Predicted_Labels.npy"
    ),
    "OOF LightGBM": load_prediction_file(
        CIFAR_PRED_DIR,
        "OOF_LightGBM_Predicted_Labels.npy"
    )
}

cifar_comparisons = [
    ("Soft Voting", "OOF LR"),
    ("Soft Voting", "OOF LightGBM"),
    ("Hold-out LR", "OOF LR"),
    (
        "Hold-out LightGBM",
        "OOF LightGBM"
    ),
    ("OOF LR", "OOF LightGBM")
]

cifar_test_rows = []

for method_a, method_b in cifar_comparisons:
    cifar_test_rows.append(
        run_mcnemar_test(
            y_true=cifar_true,
            pred_a=cifar_predictions[method_a],
            pred_b=cifar_predictions[method_b],
            method_a=method_a,
            method_b=method_b,
            dataset="CIFAR-10"
        )
    )

cifar_mcnemar_results = pd.DataFrame(
    cifar_test_rows
)

display(cifar_mcnemar_results)

In [ ]:
all_mcnemar_results = pd.concat(
    [
        ct_mcnemar_results,
        cifar_mcnemar_results
    ],
    ignore_index=True
)

all_mcnemar_results.to_csv(
    os.path.join(
        FINAL_DIR,
        "Paper1_McNemar_Test_Results.csv"
    ),
    index=False
)

display(all_mcnemar_results)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score
)

BOOTSTRAP_ITERATIONS = 5000
BOOTSTRAP_SEED = 42
CONFIDENCE_LEVEL = 0.95

print("Bootstrap iterations:", BOOTSTRAP_ITERATIONS)
print("Confidence level:", CONFIDENCE_LEVEL)

In [ ]:
def stratified_bootstrap_indices(
    y_true,
    rng
):
    """
    Generate one stratified bootstrap sample.

    Samples with replacement separately within each class,
    preserving the original number of observations per class.
    """

    y_true = np.asarray(y_true).reshape(-1)

    sampled_indices = []

    for class_label in np.unique(y_true):

        class_indices = np.where(
            y_true == class_label
        )[0]

        class_sample = rng.choice(
            class_indices,
            size=len(class_indices),
            replace=True
        )

        sampled_indices.append(
            class_sample
        )

    sampled_indices = np.concatenate(
        sampled_indices
    )

    rng.shuffle(sampled_indices)

    return sampled_indices

In [ ]:
def bootstrap_metric_confidence_intervals(
    y_true,
    probabilities,
    dataset_name,
    method_name,
    n_iterations=5000,
    confidence_level=0.95,
    random_state=42
):
    """
    Calculate stratified bootstrap 95% confidence intervals for:

    - Accuracy
    - Macro F1
    - ROC-AUC

    Supports binary and multiclass probability matrices.
    """

    y_true = np.asarray(
        y_true
    ).reshape(-1)

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    )

    if probabilities.ndim != 2:
        raise ValueError(
            f"{method_name}: probabilities must be a 2D array."
        )

    if len(y_true) != len(probabilities):
        raise ValueError(
            f"{method_name}: label and probability lengths differ."
        )

    if not np.isfinite(probabilities).all():
        raise ValueError(
            f"{method_name}: probabilities contain NaN or infinity."
        )

    predicted_labels = np.argmax(
        probabilities,
        axis=1
    )

    number_of_classes = probabilities.shape[1]

    rng = np.random.default_rng(
        random_state
    )

    bootstrap_accuracy = []
    bootstrap_macro_f1 = []
    bootstrap_auc = []

    for _ in range(n_iterations):

        sampled_indices = stratified_bootstrap_indices(
            y_true,
            rng
        )

        sampled_true = y_true[
            sampled_indices
        ]

        sampled_predicted = predicted_labels[
            sampled_indices
        ]

        sampled_probabilities = probabilities[
            sampled_indices
        ]

        bootstrap_accuracy.append(
            accuracy_score(
                sampled_true,
                sampled_predicted
            )
        )

        bootstrap_macro_f1.append(
            f1_score(
                sampled_true,
                sampled_predicted,
                average="macro",
                zero_division=0
            )
        )

        if number_of_classes == 2:

            auc_value = roc_auc_score(
                sampled_true,
                sampled_probabilities[:, 1]
            )

        else:

            auc_value = roc_auc_score(
                sampled_true,
                sampled_probabilities,
                multi_class="ovr",
                average="macro"
            )

        bootstrap_auc.append(
            auc_value
        )

    alpha = 1.0 - confidence_level

    lower_percentile = (
        100 * alpha / 2
    )

    upper_percentile = (
        100 * (1 - alpha / 2)
    )

    point_accuracy = accuracy_score(
        y_true,
        predicted_labels
    )

    point_macro_f1 = f1_score(
        y_true,
        predicted_labels,
        average="macro",
        zero_division=0
    )

    if number_of_classes == 2:

        point_auc = roc_auc_score(
            y_true,
            probabilities[:, 1]
        )

    else:

        point_auc = roc_auc_score(
            y_true,
            probabilities,
            multi_class="ovr",
            average="macro"
        )

    return {
        "Dataset": dataset_name,
        "Method": method_name,

        "Accuracy": point_accuracy,
        "Accuracy_CI_Lower": np.percentile(
            bootstrap_accuracy,
            lower_percentile
        ),
        "Accuracy_CI_Upper": np.percentile(
            bootstrap_accuracy,
            upper_percentile
        ),

        "Macro_F1": point_macro_f1,
        "Macro_F1_CI_Lower": np.percentile(
            bootstrap_macro_f1,
            lower_percentile
        ),
        "Macro_F1_CI_Upper": np.percentile(
            bootstrap_macro_f1,
            upper_percentile
        ),

        "ROC_AUC": point_auc,
        "ROC_AUC_CI_Lower": np.percentile(
            bootstrap_auc,
            lower_percentile
        ),
        "ROC_AUC_CI_Upper": np.percentile(
            bootstrap_auc,
            upper_percentile
        ),

        "Bootstrap_Iterations": n_iterations,
        "Confidence_Level": confidence_level
    }

In [ ]:
ct_probability_files = {
    "Soft Voting": (
        "SoftVoting_Test_Probabilities.npy"
    ),
    "Hold-out LR": (
        "Holdout_LogisticRegression_Test_Probabilities.npy"
    ),
    "Hold-out LightGBM": (
        "Holdout_LightGBM_Test_Probabilities.npy"
    ),
    "OOF LR": (
        "OOF_LogisticRegression_Test_Probabilities.npy"
    ),
    "OOF LightGBM": (
        "OOF_LightGBM_Test_Probabilities.npy"
    )
}

ct_true_path = os.path.join(
    CT_PRED_DIR,
    "CT_Test_True_Labels.npy"
)

if not os.path.exists(ct_true_path):
    raise FileNotFoundError(
        f"CT true labels not found:\n{ct_true_path}"
    )

ct_true_labels = np.load(
    ct_true_path
).reshape(-1)

ct_probability_outputs = {}

for method_name, filename in ct_probability_files.items():

    file_path = os.path.join(
        CT_PRED_DIR,
        filename
    )

    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Missing CT probability file:\n{file_path}"
        )

    ct_probability_outputs[
        method_name
    ] = np.load(
        file_path
    ).astype(np.float32)

    print(
        method_name,
        ct_probability_outputs[
            method_name
        ].shape
    )

In [ ]:
ct_bootstrap_rows = []

for method_name, probabilities in (
    ct_probability_outputs.items()
):

    print(
        "Running CT bootstrap:",
        method_name
    )

    result = (
        bootstrap_metric_confidence_intervals(
            y_true=ct_true_labels,
            probabilities=probabilities,
            dataset_name="CT",
            method_name=method_name,
            n_iterations=BOOTSTRAP_ITERATIONS,
            confidence_level=CONFIDENCE_LEVEL,
            random_state=BOOTSTRAP_SEED
        )
    )

    ct_bootstrap_rows.append(
        result
    )

ct_bootstrap_results = pd.DataFrame(
    ct_bootstrap_rows
)

display(
    ct_bootstrap_results.round(6)
)

In [ ]:
cifar_probability_files = {
    "Soft Voting": (
        "SoftVoting_Test_Probabilities.npy"
    ),
    "Hold-out LR": (
        "Holdout_LogisticRegression_Test_Probabilities.npy"
    ),
    "Hold-out LightGBM": (
        "Holdout_LightGBM_Test_Probabilities.npy"
    ),
    "OOF LR": (
        "OOF_LogisticRegression_Test_Probabilities.npy"
    ),
    "OOF LightGBM": (
        "OOF_LightGBM_Test_Probabilities.npy"
    )
}

cifar_true_path = os.path.join(
    CIFAR_PRED_DIR,
    "CIFAR10_Test_True_Labels.npy"
)

if not os.path.exists(cifar_true_path):
    raise FileNotFoundError(
        "CIFAR-10 true labels not found:\n"
        + cifar_true_path
    )

cifar_true_labels = np.load(
    cifar_true_path
).reshape(-1)

cifar_probability_outputs = {}

for method_name, filename in (
    cifar_probability_files.items()
):

    file_path = os.path.join(
        CIFAR_PRED_DIR,
        filename
    )

    if not os.path.exists(file_path):
        raise FileNotFoundError(
            "Missing CIFAR-10 probability file:\n"
            + file_path
        )

    cifar_probability_outputs[
        method_name
    ] = np.load(
        file_path
    ).astype(np.float32)

    print(
        method_name,
        cifar_probability_outputs[
            method_name
        ].shape
    )

In [ ]:
cifar_bootstrap_rows = []

for method_name, probabilities in (
    cifar_probability_outputs.items()
):

    print(
        "Running CIFAR-10 bootstrap:",
        method_name
    )

    result = (
        bootstrap_metric_confidence_intervals(
            y_true=cifar_true_labels,
            probabilities=probabilities,
            dataset_name="CIFAR-10",
            method_name=method_name,
            n_iterations=BOOTSTRAP_ITERATIONS,
            confidence_level=CONFIDENCE_LEVEL,
            random_state=BOOTSTRAP_SEED
        )
    )

    cifar_bootstrap_rows.append(
        result
    )

cifar_bootstrap_results = pd.DataFrame(
    cifar_bootstrap_rows
)

display(
    cifar_bootstrap_results.round(6)
)

In [ ]:
all_bootstrap_results = pd.concat(
    [
        ct_bootstrap_results,
        cifar_bootstrap_results
    ],
    ignore_index=True
)

bootstrap_csv_path = os.path.join(
    FINAL_DIR,
    "Paper1_Bootstrap_95CI_Results.csv"
)

all_bootstrap_results.to_csv(
    bootstrap_csv_path,
    index=False
)

display(
    all_bootstrap_results.round(6)
)

print(
    "Saved:",
    bootstrap_csv_path
)

In [ ]:
bootstrap_paper_table = (
    all_bootstrap_results.copy()
)

percentage_columns = [
    "Accuracy",
    "Accuracy_CI_Lower",
    "Accuracy_CI_Upper",
    "Macro_F1",
    "Macro_F1_CI_Lower",
    "Macro_F1_CI_Upper",
    "ROC_AUC",
    "ROC_AUC_CI_Lower",
    "ROC_AUC_CI_Upper"
]

for column in percentage_columns:
    bootstrap_paper_table[column] = (
        bootstrap_paper_table[column]
        * 100
    )

bootstrap_paper_table[
    "Accuracy_95CI"
] = bootstrap_paper_table.apply(
    lambda row: (
        f"{row['Accuracy']:.2f}% "
        f"[{row['Accuracy_CI_Lower']:.2f}, "
        f"{row['Accuracy_CI_Upper']:.2f}]"
    ),
    axis=1
)

bootstrap_paper_table[
    "Macro_F1_95CI"
] = bootstrap_paper_table.apply(
    lambda row: (
        f"{row['Macro_F1']:.2f}% "
        f"[{row['Macro_F1_CI_Lower']:.2f}, "
        f"{row['Macro_F1_CI_Upper']:.2f}]"
    ),
    axis=1
)

bootstrap_paper_table[
    "ROC_AUC_95CI"
] = bootstrap_paper_table.apply(
    lambda row: (
        f"{row['ROC_AUC']:.2f}% "
        f"[{row['ROC_AUC_CI_Lower']:.2f}, "
        f"{row['ROC_AUC_CI_Upper']:.2f}]"
    ),
    axis=1
)

bootstrap_summary = bootstrap_paper_table[
    [
        "Dataset",
        "Method",
        "Accuracy_95CI",
        "Macro_F1_95CI",
        "ROC_AUC_95CI"
    ]
]

bootstrap_summary_path = os.path.join(
    FINAL_DIR,
    "Paper1_Bootstrap_95CI_Paper_Table.csv"
)

bootstrap_summary.to_csv(
    bootstrap_summary_path,
    index=False
)

display(bootstrap_summary)

print(
    "Saved:",
    bootstrap_summary_path
)

In [ ]:
plot_df = all_bootstrap_results.copy()

plot_df["Accuracy_Percent"] = (
    plot_df["Accuracy"] * 100
)

plot_df["Lower_Error"] = (
    plot_df["Accuracy"]
    - plot_df["Accuracy_CI_Lower"]
) * 100

plot_df["Upper_Error"] = (
    plot_df["Accuracy_CI_Upper"]
    - plot_df["Accuracy"]
) * 100

plot_df["Label"] = (
    plot_df["Dataset"]
    + "\n"
    + plot_df["Method"]
)

x_positions = np.arange(
    len(plot_df)
)

plt.figure(figsize=(14, 7))

bars = plt.bar(
    x_positions,
    plot_df["Accuracy_Percent"]
)

plt.errorbar(
    x_positions,
    plot_df["Accuracy_Percent"],
    yerr=np.vstack(
        [
            plot_df["Lower_Error"],
            plot_df["Upper_Error"]
        ]
    ),
    fmt="none",
    capsize=5,
    linewidth=1.5
)

for bar, value in zip(
    bars,
    plot_df["Accuracy_Percent"]
):
    plt.text(
        bar.get_x()
        + bar.get_width() / 2,
        bar.get_height() + 0.12,
        f"{value:.2f}%",
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

plt.xticks(
    x_positions,
    plot_df["Label"],
    rotation=30,
    ha="right"
)

plt.ylabel("Accuracy (%)")
plt.xlabel("Dataset and ensemble method")
plt.title(
    "Accuracy with stratified-bootstrap 95% confidence intervals"
)

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.4
)

minimum_limit = (
    plot_df["Accuracy_Percent"]
    - plot_df["Lower_Error"]
).min() - 1

maximum_limit = (
    plot_df["Accuracy_Percent"]
    + plot_df["Upper_Error"]
).max() + 1

plt.ylim(
    minimum_limit,
    min(100, maximum_limit)
)

plt.tight_layout()

bootstrap_figure_path = os.path.join(
    FINAL_DIR,
    "Paper1_Accuracy_Bootstrap_95CI.png"
)

plt.savefig(
    bootstrap_figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Saved:",
    bootstrap_figure_path
)

In [ ]:
from sklearn.metrics import log_loss, brier_score_loss
from sklearn.calibration import calibration_curve

CALIBRATION_BINS = 15

In [ ]:
def expected_calibration_error(
    y_true,
    probabilities,
    n_bins=15
):
    """
    Calculate multiclass Expected Calibration Error using
    maximum predicted confidence.
    """

    y_true = np.asarray(y_true).reshape(-1)
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    )

    predicted_labels = np.argmax(
        probabilities,
        axis=1
    )

    confidences = np.max(
        probabilities,
        axis=1
    )

    correct = (
        predicted_labels == y_true
    ).astype(np.float64)

    bin_edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1
    )

    ece = 0.0

    for bin_index in range(n_bins):

        lower = bin_edges[bin_index]
        upper = bin_edges[bin_index + 1]

        if bin_index == n_bins - 1:
            in_bin = (
                (confidences >= lower)
                & (confidences <= upper)
            )
        else:
            in_bin = (
                (confidences >= lower)
                & (confidences < upper)
            )

        proportion = np.mean(in_bin)

        if proportion > 0:

            bin_accuracy = np.mean(
                correct[in_bin]
            )

            bin_confidence = np.mean(
                confidences[in_bin]
            )

            ece += proportion * abs(
                bin_accuracy
                - bin_confidence
            )

    return float(ece)

In [ ]:
def multiclass_brier_score(
    y_true,
    probabilities
):
    """
    Mean squared difference between one-hot labels
    and predicted class probabilities.
    """

    y_true = np.asarray(
        y_true
    ).reshape(-1)

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    )

    number_of_classes = (
        probabilities.shape[1]
    )

    one_hot_labels = np.eye(
        number_of_classes
    )[y_true]

    return float(
        np.mean(
            np.sum(
                (
                    probabilities
                    - one_hot_labels
                ) ** 2,
                axis=1
            )
        )
    )

In [ ]:
def evaluate_calibration(
    y_true,
    probabilities,
    dataset_name,
    method_name,
    n_bins=15
):
    probabilities = np.asarray(
        probabilities,
        dtype=np.float64
    )

    probabilities = (
        probabilities
        / probabilities.sum(
            axis=1,
            keepdims=True
        )
    )

    return {
        "Dataset": dataset_name,
        "Method": method_name,
        "Log_Loss": log_loss(
            y_true,
            probabilities
        ),
        "Brier_Score": (
            multiclass_brier_score(
                y_true,
                probabilities
            )
        ),
        "ECE": expected_calibration_error(
            y_true,
            probabilities,
            n_bins=n_bins
        )
    }

In [ ]:
ct_calibration_rows = []

for method_name, probabilities in (
    ct_probability_outputs.items()
):

    ct_calibration_rows.append(
        evaluate_calibration(
            y_true=ct_true_labels,
            probabilities=probabilities,
            dataset_name="CT",
            method_name=method_name,
            n_bins=CALIBRATION_BINS
        )
    )

ct_calibration_results = pd.DataFrame(
    ct_calibration_rows
)

display(
    ct_calibration_results.round(6)
)

In [ ]:
cifar_calibration_rows = []

for method_name, probabilities in (
    cifar_probability_outputs.items()
):

    cifar_calibration_rows.append(
        evaluate_calibration(
            y_true=cifar_true_labels,
            probabilities=probabilities,
            dataset_name="CIFAR-10",
            method_name=method_name,
            n_bins=CALIBRATION_BINS
        )
    )

cifar_calibration_results = pd.DataFrame(
    cifar_calibration_rows
)

display(
    cifar_calibration_results.round(6)
)

In [ ]:
all_calibration_results = pd.concat(
    [
        ct_calibration_results,
        cifar_calibration_results
    ],
    ignore_index=True
)

calibration_csv_path = os.path.join(
    FINAL_DIR,
    "Paper1_Calibration_Results.csv"
)

all_calibration_results.to_csv(
    calibration_csv_path,
    index=False
)

display(
    all_calibration_results.round(6)
)

print(
    "Saved:",
    calibration_csv_path
)

In [ ]:
def plot_reliability_diagram(
    y_true,
    probability_dict,
    dataset_name,
    save_path,
    n_bins=15
):
    plt.figure(figsize=(8, 7))

    for method_name, probabilities in (
        probability_dict.items()
    ):

        predicted_labels = np.argmax(
            probabilities,
            axis=1
        )

        confidences = np.max(
            probabilities,
            axis=1
        )

        correctness = (
            predicted_labels == y_true
        ).astype(int)

        fraction_correct, mean_confidence = (
            calibration_curve(
                correctness,
                confidences,
                n_bins=n_bins,
                strategy="uniform"
            )
        )

        plt.plot(
            mean_confidence,
            fraction_correct,
            marker="o",
            label=method_name
        )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Perfect calibration"
    )

    plt.xlabel("Mean predicted confidence")
    plt.ylabel("Observed accuracy")
    plt.title(
        f"{dataset_name} reliability diagram"
    )

    plt.legend()
    plt.grid(
        linestyle="--",
        alpha=0.4
    )

    plt.tight_layout()

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
ct_reliability_path = os.path.join(
    FINAL_DIR,
    "Paper1_CT_Reliability_Diagram.png"
)

plot_reliability_diagram(
    y_true=ct_true_labels,
    probability_dict=ct_probability_outputs,
    dataset_name="CT",
    save_path=ct_reliability_path,
    n_bins=CALIBRATION_BINS
)

print("Saved:", ct_reliability_path)

In [ ]:
cifar_reliability_path = os.path.join(
    FINAL_DIR,
    "Paper1_CIFAR10_Reliability_Diagram.png"
)

plot_reliability_diagram(
    y_true=cifar_true_labels,
    probability_dict=cifar_probability_outputs,
    dataset_name="CIFAR-10",
    save_path=cifar_reliability_path,
    n_bins=CALIBRATION_BINS
)

print("Saved:", cifar_reliability_path)

In [ ]:
from itertools import combinations

def prediction_labels(probabilities):
    return np.argmax(
        probabilities,
        axis=1
    )


def pairwise_diversity_metrics(
    y_true,
    probabilities_a,
    probabilities_b,
    model_a,
    model_b,
    dataset_name
):
    y_true = np.asarray(
        y_true
    ).reshape(-1)

    pred_a = prediction_labels(
        probabilities_a
    )

    pred_b = prediction_labels(
        probabilities_b
    )

    correct_a = pred_a == y_true
    correct_b = pred_b == y_true

    n11 = int(
        np.sum(
            correct_a & correct_b
        )
    )

    n10 = int(
        np.sum(
            correct_a & (~correct_b)
        )
    )

    n01 = int(
        np.sum(
            (~correct_a) & correct_b
        )
    )

    n00 = int(
        np.sum(
            (~correct_a) & (~correct_b)
        )
    )

    total = len(y_true)

    disagreement = (
        n10 + n01
    ) / total

    double_fault = n00 / total

    denominator_q = (
        n11 * n00
        + n10 * n01
    )

    if denominator_q == 0:
        q_statistic = np.nan
    else:
        q_statistic = (
            n11 * n00
            - n10 * n01
        ) / denominator_q

    denominator_phi = np.sqrt(
        (n11 + n10)
        * (n01 + n00)
        * (n11 + n01)
        * (n10 + n00)
    )

    if denominator_phi == 0:
        error_correlation = np.nan
    else:
        error_correlation = (
            n11 * n00
            - n10 * n01
        ) / denominator_phi

    return {
        "Dataset": dataset_name,
        "Model_A": model_a,
        "Model_B": model_b,
        "N11_Both_Correct": n11,
        "N10_A_Correct_B_Wrong": n10,
        "N01_A_Wrong_B_Correct": n01,
        "N00_Both_Wrong": n00,
        "Disagreement_Rate": disagreement,
        "Double_Fault": double_fault,
        "Q_Statistic": q_statistic,
        "Error_Correlation": error_correlation
    }

In [ ]:
# ============================================================
# CELL 31 — PAIRWISE DIVERSITY METRIC FUNCTIONS
# ============================================================

import os
import numpy as np
import pandas as pd
from itertools import combinations


def prediction_labels(probabilities):
    """
    Convert class-probability matrices into predicted labels.
    """

    probabilities = np.asarray(
        probabilities
    )

    if probabilities.ndim != 2:
        raise ValueError(
            "Probability input must be a 2D array."
        )

    return np.argmax(
        probabilities,
        axis=1
    )


def pairwise_diversity_metrics(
    y_true,
    probabilities_a,
    probabilities_b,
    model_a,
    model_b,
    dataset_name
):
    """
    Calculate pairwise ensemble-diversity measures.

    N11: both models correct
    N10: model A correct, model B wrong
    N01: model A wrong, model B correct
    N00: both models wrong
    """

    y_true = np.asarray(
        y_true
    ).reshape(-1)

    probabilities_a = np.asarray(
        probabilities_a
    )

    probabilities_b = np.asarray(
        probabilities_b
    )

    if len(probabilities_a) != len(y_true):
        raise ValueError(
            f"{model_a}: probability and label lengths differ."
        )

    if len(probabilities_b) != len(y_true):
        raise ValueError(
            f"{model_b}: probability and label lengths differ."
        )

    pred_a = prediction_labels(
        probabilities_a
    )

    pred_b = prediction_labels(
        probabilities_b
    )

    correct_a = pred_a == y_true
    correct_b = pred_b == y_true

    n11 = int(
        np.sum(
            correct_a & correct_b
        )
    )

    n10 = int(
        np.sum(
            correct_a & (~correct_b)
        )
    )

    n01 = int(
        np.sum(
            (~correct_a) & correct_b
        )
    )

    n00 = int(
        np.sum(
            (~correct_a) & (~correct_b)
        )
    )

    total = len(y_true)

    disagreement_rate = (
        n10 + n01
    ) / total

    double_fault = (
        n00 / total
    )

    q_denominator = (
        n11 * n00
        + n10 * n01
    )

    if q_denominator == 0:
        q_statistic = np.nan
    else:
        q_statistic = (
            n11 * n00
            - n10 * n01
        ) / q_denominator

    correlation_denominator = np.sqrt(
        (n11 + n10)
        * (n01 + n00)
        * (n11 + n01)
        * (n10 + n00)
    )

    if correlation_denominator == 0:
        error_correlation = np.nan
    else:
        error_correlation = (
            n11 * n00
            - n10 * n01
        ) / correlation_denominator

    oracle_correct = (
        correct_a | correct_b
    )

    oracle_accuracy = np.mean(
        oracle_correct
    )

    probability_difference = np.mean(
        np.abs(
            probabilities_a
            - probabilities_b
        )
    )

    return {
        "Dataset": dataset_name,
        "Model_A": model_a,
        "Model_B": model_b,
        "N11_Both_Correct": n11,
        "N10_A_Correct_B_Wrong": n10,
        "N01_A_Wrong_B_Correct": n01,
        "N00_Both_Wrong": n00,
        "Disagreement_Rate": disagreement_rate,
        "Double_Fault": double_fault,
        "Q_Statistic": q_statistic,
        "Error_Correlation": error_correlation,
        "Oracle_Accuracy": oracle_accuracy,
        "Mean_Probability_Difference": (
            probability_difference
        )
    }


print("Cell 31 loaded successfully.")
print("Diversity functions are ready.")

In [ ]:
# ============================================================
# CELL 32 — LOAD CT TRUE LABELS AND BASE-MODEL PROBABILITIES
# ============================================================

import os
import numpy as np

BASE_DIR = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison"
)

CT_PRED_DIR = os.path.join(
    BASE_DIR,
    "CT",
    "Test_Predictions"
)

CT_TRUE_LABELS_PATH = os.path.join(
    CT_PRED_DIR,
    "CT_Test_True_Labels.npy"
)

if not os.path.exists(CT_TRUE_LABELS_PATH):
    raise FileNotFoundError(
        "CT true-label file not found:\n"
        + CT_TRUE_LABELS_PATH
    )

ct_true_labels = np.load(
    CT_TRUE_LABELS_PATH
).reshape(-1).astype(np.int64)

print(
    "CT true labels loaded:",
    ct_true_labels.shape
)

CT_OOF_PROB_DIR = (
    "/content/drive/MyDrive/"
    "CT_OOF_Stacking_Results/"
    "OOF_Probabilities"
)

ct_file_paths = {
    "ResNet50": os.path.join(
        CT_OOF_PROB_DIR,
        "CT_ResNet50_Test_Probabilities.npy"
    ),

    "EfficientNetB0": os.path.join(
        CT_OOF_PROB_DIR,
        "CT_EfficientNetB0_Test_Probabilities.npy"
    )
}

ct_base_probabilities = {}

for model_name, file_path in ct_file_paths.items():

    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Missing CT probability file:\n{file_path}"
        )

    probabilities = np.load(
        file_path
    ).astype(np.float32)

    expected_shape = (
        len(ct_true_labels),
        2
    )

    if probabilities.shape != expected_shape:
        raise ValueError(
            f"{model_name}: obtained "
            f"{probabilities.shape}, expected "
            f"{expected_shape}."
        )

    if not np.isfinite(
        probabilities
    ).all():
        raise ValueError(
            f"{model_name}: probabilities contain "
            "NaN or infinity."
        )

    ct_base_probabilities[
        model_name
    ] = probabilities

    print("\nModel:", model_name)
    print("File:", file_path)
    print("Shape:", probabilities.shape)

print(
    "\nCT labels and base-model probabilities "
    "loaded successfully."
)

In [ ]:
# ============================================================
# CELL 33 — CALCULATE CT PAIRWISE DIVERSITY
# ============================================================

ct_diversity_rows = []

ct_model_names = list(
    ct_base_probabilities.keys()
)

for model_a, model_b in combinations(
    ct_model_names,
    2
):
    result = pairwise_diversity_metrics(
        y_true=ct_true_labels,
        probabilities_a=ct_base_probabilities[
            model_a
        ],
        probabilities_b=ct_base_probabilities[
            model_b
        ],
        model_a=model_a,
        model_b=model_b,
        dataset_name="CT"
    )

    ct_diversity_rows.append(
        result
    )

ct_diversity_results = pd.DataFrame(
    ct_diversity_rows
)

display(
    ct_diversity_results.round(6)
)

# Save CT diversity results
CT_DIVERSITY_PATH = os.path.join(
    FINAL_DIR,
    "Paper1_CT_Base_Model_Diversity.csv"
)

ct_diversity_results.to_csv(
    CT_DIVERSITY_PATH,
    index=False
)

print(
    "\nSaved:",
    CT_DIVERSITY_PATH
)

In [ ]:
# ============================================================
# SAVE CT DIVERSITY RESULTS
# ============================================================

import os

BASE_DIR = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison"
)

FINAL_DIR = os.path.join(
    BASE_DIR,
    "Combined_Paper_Results"
)

os.makedirs(
    FINAL_DIR,
    exist_ok=True
)

CT_DIVERSITY_PATH = os.path.join(
    FINAL_DIR,
    "Paper1_CT_Base_Model_Diversity.csv"
)

ct_diversity_results.to_csv(
    CT_DIVERSITY_PATH,
    index=False
)

print("Saved:", CT_DIVERSITY_PATH)

In [ ]:
# ============================================================
# CELL 34 — LOAD CIFAR-10 TRUE LABELS AND BASE PROBABILITIES
# ============================================================

import os
import numpy as np

BASE_DIR = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison"
)

CIFAR_PRED_DIR = os.path.join(
    BASE_DIR,
    "CIFAR10",
    "Test_Predictions"
)

CIFAR_TRUE_LABELS_PATH = os.path.join(
    CIFAR_PRED_DIR,
    "CIFAR10_Test_True_Labels.npy"
)

if not os.path.exists(CIFAR_TRUE_LABELS_PATH):
    raise FileNotFoundError(
        "CIFAR-10 true-label file not found:\n"
        + CIFAR_TRUE_LABELS_PATH
    )

cifar_true_labels = np.load(
    CIFAR_TRUE_LABELS_PATH
).reshape(-1).astype(np.int64)

print(
    "CIFAR-10 true labels loaded:",
    cifar_true_labels.shape
)

CIFAR_OOF_PROB_DIR = (
    "/content/drive/MyDrive/"
    "OOF_Stacking_Results/"
    "OOF_Probabilities"
)

cifar_file_paths = {
    "CNN": os.path.join(
        CIFAR_OOF_PROB_DIR,
        "CIFAR_CNN_Test_Probabilities.npy"
    ),
    "VGG16": os.path.join(
        CIFAR_OOF_PROB_DIR,
        "CIFAR_VGG16_Test_Probabilities.npy"
    ),
    "ResNet50": os.path.join(
        CIFAR_OOF_PROB_DIR,
        "CIFAR_ResNet50_Test_Probabilities.npy"
    ),
    "EfficientNetB0": os.path.join(
        CIFAR_OOF_PROB_DIR,
        "CIFAR_EfficientNetB0_Test_Probabilities.npy"
    )
}

cifar_base_probabilities = {}

for model_name, file_path in cifar_file_paths.items():

    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"Missing CIFAR-10 probability file:\n{file_path}"
        )

    probabilities = np.load(
        file_path
    ).astype(np.float32)

    expected_shape = (
        len(cifar_true_labels),
        10
    )

    if probabilities.shape != expected_shape:
        raise ValueError(
            f"{model_name}: obtained "
            f"{probabilities.shape}, expected "
            f"{expected_shape}."
        )

    if not np.isfinite(probabilities).all():
        raise ValueError(
            f"{model_name}: probabilities contain "
            "NaN or infinity."
        )

    cifar_base_probabilities[
        model_name
    ] = probabilities

    print(
        model_name,
        probabilities.shape,
        "loaded successfully."
    )

print(
    "\nCIFAR-10 labels and base-model probabilities "
    "loaded successfully."
)

In [ ]:
# ============================================================
# CELL 35 — CALCULATE CIFAR-10 PAIRWISE DIVERSITY
# ============================================================

cifar_diversity_rows = []

cifar_model_names = list(
    cifar_base_probabilities.keys()
)

for model_a, model_b in combinations(
    cifar_model_names,
    2
):
    result = pairwise_diversity_metrics(
        y_true=cifar_true_labels,
        probabilities_a=cifar_base_probabilities[
            model_a
        ],
        probabilities_b=cifar_base_probabilities[
            model_b
        ],
        model_a=model_a,
        model_b=model_b,
        dataset_name="CIFAR-10"
    )

    cifar_diversity_rows.append(
        result
    )

cifar_diversity_results = pd.DataFrame(
    cifar_diversity_rows
)

display(
    cifar_diversity_results.round(6)
)

In [ ]:
# ============================================================
# CELL 36 — COMBINE AND SAVE DIVERSITY RESULTS
# ============================================================

import os
import pandas as pd

BASE_DIR = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison"
)

FINAL_DIR = os.path.join(
    BASE_DIR,
    "Combined_Paper_Results"
)

os.makedirs(
    FINAL_DIR,
    exist_ok=True
)

all_diversity_results = pd.concat(
    [
        ct_diversity_results,
        cifar_diversity_results
    ],
    ignore_index=True
)

DIVERSITY_PATH = os.path.join(
    FINAL_DIR,
    "Paper1_Base_Model_Diversity.csv"
)

all_diversity_results.to_csv(
    DIVERSITY_PATH,
    index=False
)

display(
    all_diversity_results.round(6)
)

print(
    "\nSaved:",
    DIVERSITY_PATH
)

In [ ]:
# ============================================================
# CELL 37 — CIFAR-10 DISAGREEMENT HEATMAP
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

# Ensure output folder exists
BASE_DIR = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison"
)

FINAL_DIR = os.path.join(
    BASE_DIR,
    "Combined_Paper_Results"
)

os.makedirs(
    FINAL_DIR,
    exist_ok=True
)

cifar_model_names = list(
    cifar_base_probabilities.keys()
)

number_of_models = len(
    cifar_model_names
)

disagreement_matrix = np.zeros(
    (
        number_of_models,
        number_of_models
    ),
    dtype=np.float64
)

# Convert all probability matrices to predicted labels
cifar_predicted_labels = {
    model_name: np.argmax(
        probabilities,
        axis=1
    )
    for model_name, probabilities
    in cifar_base_probabilities.items()
}

# Calculate pairwise disagreement
for i, model_a in enumerate(
    cifar_model_names
):
    for j, model_b in enumerate(
        cifar_model_names
    ):
        disagreement_matrix[i, j] = np.mean(
            cifar_predicted_labels[model_a]
            != cifar_predicted_labels[model_b]
        )

print("CIFAR-10 disagreement matrix:")
print(
    np.round(
        disagreement_matrix,
        4
    )
)

# Create heatmap
plt.figure(
    figsize=(8, 7)
)

heatmap = plt.imshow(
    disagreement_matrix,
    vmin=0.0,
    vmax=max(
        0.12,
        disagreement_matrix.max()
    )
)

colorbar = plt.colorbar(
    heatmap
)

colorbar.set_label(
    "Disagreement rate"
)

plt.xticks(
    np.arange(number_of_models),
    cifar_model_names,
    rotation=30,
    ha="right"
)

plt.yticks(
    np.arange(number_of_models),
    cifar_model_names
)

# Add values inside cells
for i in range(
    number_of_models
):
    for j in range(
        number_of_models
    ):
        plt.text(
            j,
            i,
            f"{disagreement_matrix[i, j]:.4f}",
            ha="center",
            va="center",
            fontsize=10
        )

plt.xlabel("Base model")
plt.ylabel("Base model")

plt.title(
    "CIFAR-10 base-model disagreement matrix"
)

plt.tight_layout()

CIFAR_DISAGREEMENT_PATH = os.path.join(
    FINAL_DIR,
    "Paper1_CIFAR10_Disagreement_Heatmap.png"
)

plt.savefig(
    CIFAR_DISAGREEMENT_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(
    "Saved:",
    CIFAR_DISAGREEMENT_PATH
)

In [ ]:
# ============================================================
# CELL 38 — LOAD SAVED META-FEATURE MATRICES AND LABELS
# ============================================================

import os
import numpy as np
import pandas as pd

BASE_DIR = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison"
)

# ----------------------------
# CIFAR-10 saved matrices
# ----------------------------

CIFAR_HOLDOUT_DIR = os.path.join(
    BASE_DIR,
    "CIFAR10",
    "Holdout_Probabilities"
)

CIFAR_PRED_DIR = os.path.join(
    BASE_DIR,
    "CIFAR10",
    "Test_Predictions"
)

CIFAR_OOF_DIR = (
    "/content/drive/MyDrive/"
    "OOF_Stacking_Results/"
    "OOF_Probabilities"
)

cifar_holdout_meta_path = os.path.join(
    CIFAR_HOLDOUT_DIR,
    "CIFAR10_Holdout_Meta_40_Features.npy"
)

cifar_holdout_labels_path = os.path.join(
    CIFAR_HOLDOUT_DIR,
    "CIFAR10_Holdout_Meta_Labels.npy"
)

cifar_test_labels_path = os.path.join(
    CIFAR_PRED_DIR,
    "CIFAR10_Test_True_Labels.npy"
)

required_cifar_paths = [
    cifar_holdout_meta_path,
    cifar_holdout_labels_path,
    cifar_test_labels_path
]

for path in required_cifar_paths:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing CIFAR-10 file:\n{path}"
        )

X_cifar_holdout_meta = np.load(
    cifar_holdout_meta_path
).astype(np.float32)

y_cifar_holdout_meta = np.load(
    cifar_holdout_labels_path
).reshape(-1).astype(np.int64)

y_cifar_test = np.load(
    cifar_test_labels_path
).reshape(-1).astype(np.int64)

cifar_model_order = [
    "CNN",
    "VGG16",
    "ResNet50",
    "EfficientNetB0"
]

cifar_oof_files = {
    "CNN": "CIFAR_CNN_5Fold_OOF.npy",
    "VGG16": "CIFAR_VGG16_5Fold_OOF.npy",
    "ResNet50": "CIFAR_ResNet50_5Fold_OOF.npy",
    "EfficientNetB0": "CIFAR_EfficientNetB0_5Fold_OOF.npy"
}

cifar_test_files = {
    "CNN": "CIFAR_CNN_Test_Probabilities.npy",
    "VGG16": "CIFAR_VGG16_Test_Probabilities.npy",
    "ResNet50": "CIFAR_ResNet50_Test_Probabilities.npy",
    "EfficientNetB0": "CIFAR_EfficientNetB0_Test_Probabilities.npy"
}

cifar_oof_arrays = []
cifar_test_arrays = []

for model_name in cifar_model_order:

    oof_path = os.path.join(
        CIFAR_OOF_DIR,
        cifar_oof_files[model_name]
    )

    test_path = os.path.join(
        CIFAR_OOF_DIR,
        cifar_test_files[model_name]
    )

    if not os.path.exists(oof_path):
        raise FileNotFoundError(
            f"Missing CIFAR-10 OOF file:\n{oof_path}"
        )

    if not os.path.exists(test_path):
        raise FileNotFoundError(
            f"Missing CIFAR-10 test file:\n{test_path}"
        )

    cifar_oof_arrays.append(
        np.load(oof_path).astype(np.float32)
    )

    cifar_test_arrays.append(
        np.load(test_path).astype(np.float32)
    )

X_cifar_oof_meta = np.concatenate(
    cifar_oof_arrays,
    axis=1
)

X_cifar_test_meta = np.concatenate(
    cifar_test_arrays,
    axis=1
)

# CIFAR-10 development labels from Drive cache
CIFAR_CACHE_DIR = (
    "/content/drive/MyDrive/CIFAR10_Cache"
)

y_cifar_oof_meta = np.load(
    os.path.join(
        CIFAR_CACHE_DIR,
        "y_cf_train.npy"
    )
).reshape(-1).astype(np.int64)

print("CIFAR hold-out meta:", X_cifar_holdout_meta.shape)
print("CIFAR OOF meta:", X_cifar_oof_meta.shape)
print("CIFAR test meta:", X_cifar_test_meta.shape)

assert X_cifar_holdout_meta.shape == (10000, 40)
assert X_cifar_oof_meta.shape == (50000, 40)
assert X_cifar_test_meta.shape == (10000, 40)

# ----------------------------
# CT saved matrices
# ----------------------------

CT_HOLDOUT_DIR = os.path.join(
    BASE_DIR,
    "CT",
    "Holdout_Probabilities"
)

CT_PRED_DIR = os.path.join(
    BASE_DIR,
    "CT",
    "Test_Predictions"
)

CT_OOF_DIR = (
    "/content/drive/MyDrive/"
    "CT_OOF_Stacking_Results/"
    "OOF_Probabilities"
)

ct_holdout_meta_path = os.path.join(
    CT_HOLDOUT_DIR,
    "CT_Holdout_Meta_Features.npy"
)

ct_holdout_labels_path = os.path.join(
    CT_HOLDOUT_DIR,
    "CT_Holdout_Meta_Labels.npy"
)

ct_test_labels_path = os.path.join(
    CT_PRED_DIR,
    "CT_Test_True_Labels.npy"
)

for path in [
    ct_holdout_meta_path,
    ct_holdout_labels_path,
    ct_test_labels_path
]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"Missing CT file:\n{path}"
        )

X_ct_holdout_meta = np.load(
    ct_holdout_meta_path
).astype(np.float32)

y_ct_holdout_meta = np.load(
    ct_holdout_labels_path
).reshape(-1).astype(np.int64)

y_ct_test = np.load(
    ct_test_labels_path
).reshape(-1).astype(np.int64)

ct_resnet_oof = np.load(
    os.path.join(
        CT_OOF_DIR,
        "CT_ResNet50_5Fold_OOF.npy"
    )
).astype(np.float32)

ct_effnet_oof = np.load(
    os.path.join(
        CT_OOF_DIR,
        "CT_EfficientNetB0_5Fold_OOF.npy"
    )
).astype(np.float32)

ct_resnet_test = np.load(
    os.path.join(
        CT_OOF_DIR,
        "CT_ResNet50_Test_Probabilities.npy"
    )
).astype(np.float32)

ct_effnet_test = np.load(
    os.path.join(
        CT_OOF_DIR,
        "CT_EfficientNetB0_Test_Probabilities.npy"
    )
).astype(np.float32)

X_ct_oof_meta = np.concatenate(
    [
        ct_resnet_oof,
        ct_effnet_oof
    ],
    axis=1
)

X_ct_test_meta = np.concatenate(
    [
        ct_resnet_test,
        ct_effnet_test
    ],
    axis=1
)

# Reconstruct CT development labels from OOF size and saved dataset
CT_DATA_PATH = (
    "/content/drive/MyDrive/"
    "PhD_CT_Transfer_Learning/data/"
    "ct_preprocessed.pkl"
)

import pickle

with open(CT_DATA_PATH, "rb") as file:
    ct_data = pickle.load(file)

y_ct_oof_meta = np.concatenate(
    [
        np.asarray(
            ct_data["y_train"]
        ).reshape(-1),
        np.asarray(
            ct_data["y_val"]
        ).reshape(-1)
    ]
).astype(np.int64)

print("\nCT hold-out meta:", X_ct_holdout_meta.shape)
print("CT OOF meta:", X_ct_oof_meta.shape)
print("CT test meta:", X_ct_test_meta.shape)

assert X_ct_holdout_meta.shape[1] == 4
assert X_ct_oof_meta.shape[1] == 4
assert X_ct_test_meta.shape[1] == 4

print("\nAll saved meta-feature matrices loaded successfully.")

In [ ]:
# ============================================================
# CELL 39 — META-LEARNER STABILITY ACROSS RANDOM SEEDS
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    log_loss
)

from lightgbm import LGBMClassifier

SEEDS = [42, 123, 2025, 7, 99]

stability_rows = []


def evaluate_meta_model(
    dataset_name,
    protocol_name,
    method_name,
    seed,
    model,
    X_train_meta,
    y_train_meta,
    X_test_meta,
    y_test
):
    model.fit(
        X_train_meta,
        y_train_meta
    )

    test_probabilities = model.predict_proba(
        X_test_meta
    )

    test_predictions = np.argmax(
        test_probabilities,
        axis=1
    )

    number_of_classes = (
        test_probabilities.shape[1]
    )

    if number_of_classes == 2:
        auc_value = roc_auc_score(
            y_test,
            test_probabilities[:, 1]
        )
    else:
        auc_value = roc_auc_score(
            y_test,
            test_probabilities,
            multi_class="ovr",
            average="macro"
        )

    return {
        "Dataset": dataset_name,
        "Protocol": protocol_name,
        "Method": method_name,
        "Seed": seed,
        "Accuracy": accuracy_score(
            y_test,
            test_predictions
        ),
        "Macro_F1": f1_score(
            y_test,
            test_predictions,
            average="macro",
            zero_division=0
        ),
        "ROC_AUC": auc_value,
        "Log_Loss": log_loss(
            y_test,
            test_probabilities
        )
    }


for seed in SEEDS:

    print(
        f"\nRunning meta-learner seed: {seed}"
    )

    # ========================================================
    # CIFAR-10 — Fixed hold-out Logistic Regression
    # ========================================================

    cifar_holdout_lr = LogisticRegression(
        max_iter=2000,
        solver="lbfgs",
        random_state=seed,
        n_jobs=-1
    )

    stability_rows.append(
        evaluate_meta_model(
            dataset_name="CIFAR-10",
            protocol_name="Fixed hold-out",
            method_name="Logistic Regression",
            seed=seed,
            model=cifar_holdout_lr,
            X_train_meta=X_cifar_holdout_meta,
            y_train_meta=y_cifar_holdout_meta,
            X_test_meta=X_cifar_test_meta,
            y_test=y_cifar_test
        )
    )

    # ========================================================
    # CIFAR-10 — OOF Logistic Regression
    # ========================================================

    cifar_oof_lr = LogisticRegression(
        max_iter=2000,
        solver="lbfgs",
        random_state=seed,
        n_jobs=-1
    )

    stability_rows.append(
        evaluate_meta_model(
            dataset_name="CIFAR-10",
            protocol_name="Five-fold OOF",
            method_name="Logistic Regression",
            seed=seed,
            model=cifar_oof_lr,
            X_train_meta=X_cifar_oof_meta,
            y_train_meta=y_cifar_oof_meta,
            X_test_meta=X_cifar_test_meta,
            y_test=y_cifar_test
        )
    )

    # ========================================================
    # CIFAR-10 — Fixed hold-out LightGBM
    # ========================================================

    cifar_holdout_lgbm = LGBMClassifier(
        objective="multiclass",
        num_class=10,
        n_estimators=300,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=5,
        subsample=0.90,
        colsample_bytree=0.90,
        reg_alpha=0.10,
        reg_lambda=0.20,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1
    )

    stability_rows.append(
        evaluate_meta_model(
            dataset_name="CIFAR-10",
            protocol_name="Fixed hold-out",
            method_name="LightGBM",
            seed=seed,
            model=cifar_holdout_lgbm,
            X_train_meta=X_cifar_holdout_meta,
            y_train_meta=y_cifar_holdout_meta,
            X_test_meta=X_cifar_test_meta,
            y_test=y_cifar_test
        )
    )

    # ========================================================
    # CIFAR-10 — OOF LightGBM
    # ========================================================

    cifar_oof_lgbm = LGBMClassifier(
        objective="multiclass",
        num_class=10,
        n_estimators=300,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=5,
        subsample=0.90,
        colsample_bytree=0.90,
        reg_alpha=0.10,
        reg_lambda=0.20,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1
    )

    stability_rows.append(
        evaluate_meta_model(
            dataset_name="CIFAR-10",
            protocol_name="Five-fold OOF",
            method_name="LightGBM",
            seed=seed,
            model=cifar_oof_lgbm,
            X_train_meta=X_cifar_oof_meta,
            y_train_meta=y_cifar_oof_meta,
            X_test_meta=X_cifar_test_meta,
            y_test=y_cifar_test
        )
    )

    # ========================================================
    # CT — Fixed hold-out Logistic Regression
    # ========================================================

    ct_holdout_lr = LogisticRegression(
        max_iter=5000,
        solver="liblinear",
        random_state=seed
    )

    stability_rows.append(
        evaluate_meta_model(
            dataset_name="CT",
            protocol_name="Fixed hold-out",
            method_name="Logistic Regression",
            seed=seed,
            model=ct_holdout_lr,
            X_train_meta=X_ct_holdout_meta,
            y_train_meta=y_ct_holdout_meta,
            X_test_meta=X_ct_test_meta,
            y_test=y_ct_test
        )
    )

    # ========================================================
    # CT — OOF Logistic Regression
    # ========================================================

    ct_oof_lr = LogisticRegression(
        max_iter=5000,
        solver="liblinear",
        random_state=seed
    )

    stability_rows.append(
        evaluate_meta_model(
            dataset_name="CT",
            protocol_name="Five-fold OOF",
            method_name="Logistic Regression",
            seed=seed,
            model=ct_oof_lr,
            X_train_meta=X_ct_oof_meta,
            y_train_meta=y_ct_oof_meta,
            X_test_meta=X_ct_test_meta,
            y_test=y_ct_test
        )
    )

    # ========================================================
    # CT — Fixed hold-out LightGBM
    # ========================================================

    ct_holdout_lgbm = LGBMClassifier(
        objective="binary",
        n_estimators=150,
        learning_rate=0.03,
        num_leaves=7,
        max_depth=3,
        min_child_samples=30,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.20,
        reg_lambda=1.00,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1
    )

    stability_rows.append(
        evaluate_meta_model(
            dataset_name="CT",
            protocol_name="Fixed hold-out",
            method_name="LightGBM",
            seed=seed,
            model=ct_holdout_lgbm,
            X_train_meta=X_ct_holdout_meta,
            y_train_meta=y_ct_holdout_meta,
            X_test_meta=X_ct_test_meta,
            y_test=y_ct_test
        )
    )

    # ========================================================
    # CT — OOF LightGBM
    # ========================================================

    ct_oof_lgbm = LGBMClassifier(
        objective="binary",
        n_estimators=150,
        learning_rate=0.03,
        num_leaves=7,
        max_depth=3,
        min_child_samples=30,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.20,
        reg_lambda=1.00,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1
    )

    stability_rows.append(
        evaluate_meta_model(
            dataset_name="CT",
            protocol_name="Five-fold OOF",
            method_name="LightGBM",
            seed=seed,
            model=ct_oof_lgbm,
            X_train_meta=X_ct_oof_meta,
            y_train_meta=y_ct_oof_meta,
            X_test_meta=X_ct_test_meta,
            y_test=y_ct_test
        )
    )


meta_stability_results = pd.DataFrame(
    stability_rows
)

display(
    meta_stability_results.round(6)
)

In [ ]:
# ============================================================
# CELL 40 — SUMMARIZE META-LEARNER STABILITY
# ============================================================

import os
import pandas as pd

BASE_DIR = (
    "/content/drive/MyDrive/"
    "Paper_1_Matched_Protocol_Comparison"
)

FINAL_DIR = os.path.join(
    BASE_DIR,
    "Combined_Paper_Results"
)

os.makedirs(
    FINAL_DIR,
    exist_ok=True
)

stability_summary = (
    meta_stability_results
    .groupby(
        [
            "Dataset",
            "Protocol",
            "Method"
        ],
        as_index=False
    )
    .agg(
        Accuracy_Mean=(
            "Accuracy",
            "mean"
        ),
        Accuracy_SD=(
            "Accuracy",
            "std"
        ),
        Accuracy_Min=(
            "Accuracy",
            "min"
        ),
        Accuracy_Max=(
            "Accuracy",
            "max"
        ),

        Macro_F1_Mean=(
            "Macro_F1",
            "mean"
        ),
        Macro_F1_SD=(
            "Macro_F1",
            "std"
        ),

        ROC_AUC_Mean=(
            "ROC_AUC",
            "mean"
        ),
        ROC_AUC_SD=(
            "ROC_AUC",
            "std"
        ),

        Log_Loss_Mean=(
            "Log_Loss",
            "mean"
        ),
        Log_Loss_SD=(
            "Log_Loss",
            "std"
        )
    )
)

# Replace NaN SD with zero when a method is fully deterministic
stability_summary = stability_summary.fillna(0.0)

# Percentage-point versions for easier interpretation
stability_summary[
    "Accuracy_Mean_Percent"
] = (
    stability_summary[
        "Accuracy_Mean"
    ] * 100
)

stability_summary[
    "Accuracy_SD_Percentage_Points"
] = (
    stability_summary[
        "Accuracy_SD"
    ] * 100
)

stability_summary[
    "Macro_F1_Mean_Percent"
] = (
    stability_summary[
        "Macro_F1_Mean"
    ] * 100
)

stability_summary[
    "ROC_AUC_Mean_Percent"
] = (
    stability_summary[
        "ROC_AUC_Mean"
    ] * 100
)

display(
    stability_summary.round(6)
)

# Save full seed-level results
seed_level_path = os.path.join(
    FINAL_DIR,
    "Paper1_Meta_Learner_Seed_Level_Results.csv"
)

meta_stability_results.to_csv(
    seed_level_path,
    index=False
)

# Save summarized results
summary_path = os.path.join(
    FINAL_DIR,
    "Paper1_Meta_Learner_Stability_Summary.csv"
)

stability_summary.to_csv(
    summary_path,
    index=False
)

print("\nSaved seed-level results:")
print(seed_level_path)

print("\nSaved stability summary:")
print(summary_path)

In [ ]:
# ============================================================
# CELL 41 — PAPER-READY STABILITY TABLE
# ============================================================

paper_stability_table = (
    stability_summary.copy()
)

paper_stability_table[
    "Accuracy_Mean_SD"
] = paper_stability_table.apply(
    lambda row: (
        f"{row['Accuracy_Mean'] * 100:.2f}% "
        f"± {row['Accuracy_SD'] * 100:.3f}"
    ),
    axis=1
)

paper_stability_table[
    "Macro_F1_Mean_SD"
] = paper_stability_table.apply(
    lambda row: (
        f"{row['Macro_F1_Mean'] * 100:.2f}% "
        f"± {row['Macro_F1_SD'] * 100:.3f}"
    ),
    axis=1
)

paper_stability_table[
    "ROC_AUC_Mean_SD"
] = paper_stability_table.apply(
    lambda row: (
        f"{row['ROC_AUC_Mean'] * 100:.2f}% "
        f"± {row['ROC_AUC_SD'] * 100:.3f}"
    ),
    axis=1
)

paper_stability_table[
    "Log_Loss_Mean_SD"
] = paper_stability_table.apply(
    lambda row: (
        f"{row['Log_Loss_Mean']:.4f} "
        f"± {row['Log_Loss_SD']:.4f}"
    ),
    axis=1
)

paper_stability_table = paper_stability_table[
    [
        "Dataset",
        "Protocol",
        "Method",
        "Accuracy_Mean_SD",
        "Macro_F1_Mean_SD",
        "ROC_AUC_Mean_SD",
        "Log_Loss_Mean_SD"
    ]
]

paper_table_path = os.path.join(
    FINAL_DIR,
    "Paper1_Meta_Learner_Stability_Paper_Table.csv"
)

paper_stability_table.to_csv(
    paper_table_path,
    index=False
)

display(
    paper_stability_table
)

print("\nSaved:")
print(paper_table_path)

In [ ]:
# ============================================================
# SOFTWARE ENVIRONMENT
# ============================================================

import sys
import platform
import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
import lightgbm
import matplotlib
import statsmodels

print("Python:", sys.version)
print("Platform:", platform.platform())
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("LightGBM:", lightgbm.__version__)
print("Matplotlib:", matplotlib.__version__)
print("statsmodels:", statsmodels.__version__)